# Phonics Book Generator – FLUX Inference Notebook

This notebook provides an end‑to‑end workflow to generate images using the **FLUX.1-schnell** model with an optional fine‑tuned **LoRA** adapter located under `output/flux_lora/`. It follows an extended outline (imports → model load → preprocessing → inputs → single & batch inference → postprocessing → benchmarking → optional ONNX export placeholder → logging → tests).

In [ ]:
# 1) Import Dependencies and Configure Device


import os, time, math, json, csv, random, pathlib, sys


from pathlib import Path


import numpy as np


import torch


from IPython.display import display, Image as IPyImage


from PIL import Image


try:
    from diffusers import FluxPipeline
except Exception as e:
    raise RuntimeError("Please install diffusers: pip install diffusers transformers accelerate") from e



# Detect device (CUDA preferred)
if torch.cuda.is_available():
    device = torch.device("cuda")
    dtype = torch.bfloat16
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    dtype = torch.float16
else:
    device = torch.device("cpu")
    dtype = torch.float32



print(f"Device: {device}")

print(f"dtype: {dtype}")

In [ ]:
# Hugging Face Login (optional if model gated or private)
try:
    from huggingface_hub import login, whoami
except ImportError:
    login = None
    whoami = None
    print("huggingface_hub not installed; install if you need gated/private models.")

HF_TOKEN = None
# Load from ENV file if present
env_path = Path("ENV")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        if line.startswith("HF_TOKEN="):
            candidate = line.split("=", 1)[1].strip()
            if candidate and not candidate.endswith("_here"):
                HF_TOKEN = candidate
                break
# Fallback to environment variables
if HF_TOKEN is None:
    HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# Interactive prompt fallback (only if running in a tty and token missing)
if HF_TOKEN is None and sys.stdin.isatty():
    try:
        inp = input("Enter Hugging Face token (or leave blank to skip): ").strip()
        if inp:
            HF_TOKEN = inp
    except Exception:
        pass

if HF_TOKEN and login:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        info = whoami(token=HF_TOKEN) if whoami else {}
        print(f"✓ Logged into Hugging Face as: {info.get('name') or info.get('username')}")
    except Exception as e:
        print(f"Warning: Hugging Face login failed: {e}")
else:
    print("No HF token available or login function missing; proceeding without authentication.")

In [ ]:
# 2) Load Trained Model or Create Inference Session

from config import get_config

# Load project config and ensure directories exist
cfg = get_config()
try:
    cfg.ensure_directories()
except Exception as e:
    print(f"Warning ensuring directories: {e}")

# Base model (can be changed if desired)
base_model = "black-forest-labs/FLUX.1-schnell"
print("Base model:", base_model)
print("Trigger word:", cfg.trigger_word)
print("Output dir:", cfg.output_dir)
print("Model name:", cfg.model_name)

# Discover LoRA weights (optional)
lora_dir = (cfg.output_dir / cfg.model_name)
lora_files = []
if lora_dir.exists():
    lora_files = sorted(lora_dir.glob("*.safetensors"))

selected_lora = lora_files[-1] if lora_files else None
print("LoRA directory:", lora_dir)
print("Found LoRA weights:", len(lora_files))
if selected_lora:
    print("Selected LoRA:", selected_lora)
else:
    print("No LoRA found; generation will use base model only.")

In [ ]:
# 3) Define Preprocessing Pipeline (Prompts Only for Text-to-Image)

# For FLUX.1-schnell we just manage text prompts; no image preprocessing.
# Placeholder for potential future conditioning (e.g., textual inversion tokens).

def normalize_prompt(p: str) -> str:
    return p.strip()

print("Preprocessing ready (text prompts).")

In [ ]:
# 4) Prepare Sample Inputs (Editable Prompts)

# Load prompts from prompts.txt (default) or fallback to hardcoded examples
prompts_file = Path("prompts.txt")
user_prompts = []

if prompts_file.exists():
    try:
        with open(prompts_file, "r", encoding="utf-8") as f:
            user_prompts = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]
        print(f"Loaded {len(user_prompts)} prompt(s) from {prompts_file}")
    except Exception as e:
        print(f"Warning: Could not read {prompts_file}: {e}")

# Fallback to example prompts if file not found or empty
if not user_prompts:
    print(f"No prompts loaded from file; using example prompts.")
    user_prompts = [
        "reading a phonics book in a cozy library",
        "holding the letter A on a bright sunny day",
        "playing with colorful alphabet blocks",
    ]

# Normalize
user_prompts = [normalize_prompt(p) for p in user_prompts if p.strip()]
print(f"Ready with {len(user_prompts)} prompt(s).")

In [ ]:
# 5) Single-Item Inference Helper Definitions

from typing import List, Optional, Union

def prefix_prompts(prompts: List[str], trigger: Optional[str]) -> List[str]:
    if not trigger:
        return prompts
    prefix = f"photo of {trigger} "
    out = []
    for p in prompts:
        if p.lower().startswith(prefix.lower()):
            out.append(p)
        else:
            out.append(prefix + p)
    return out

_pipeline_cache = None

def load_pipeline(base_model: str, lora_path: Optional[Path]):
    global _pipeline_cache
    if _pipeline_cache is not None:
        return _pipeline_cache
    print("Loading pipeline...")
    pipe = FluxPipeline.from_pretrained(base_model, torch_dtype=dtype).to(device)
    if lora_path and lora_path.exists():
        try:
            pipe.load_lora_weights(str(lora_path))
            print(f"Applied LoRA: {lora_path}")
        except Exception as e:
            print(f"Warning applying LoRA: {e}")
    else:
        print("No LoRA applied (base model only).")
    _pipeline_cache = pipe
    return pipe

GenerationRecord = []  # will hold tuples (prompt, path, seed, elapsed)

def generate_single(
    prompt: str,
    width=1024,
    height=1024,
    steps=4,
    guidance=1.0,
    seed: Optional[int] = None,
    save: bool = True,
    skip_preprocess: bool = False,
    four_variations: bool = False,
) -> Union[Image.Image, List[Image.Image]]:
    """Generate image(s) from one prompt.

    Args:
        prompt: Text prompt. If skip_preprocess=False, it will be normalized and prefixed.
        width: Image width in pixels
        height: Image height in pixels
        steps: Number of inference steps
        guidance: Guidance scale
        seed: Random seed (random if None). For four_variations, used as base seed.
        save: Whether to save the image(s) to disk
        skip_preprocess: If True, do NOT normalize or add the trigger prefix.
        four_variations: If True, generate 4 images with sequential seeds.

    Returns:
        PIL.Image if four_variations is False, else List[PIL.Image]
    """
    # Build final prompt
    if skip_preprocess:
        final_prompt = prompt
    else:
        normalized = normalize_prompt(prompt)
        final_prompt = prefix_prompts([normalized], cfg.trigger_word)[0]

    pipe = load_pipeline(base_model, selected_lora)
    out_dir = cfg.output_dir / "generated_images"
    out_dir.mkdir(parents=True, exist_ok=True)

    base_seed = seed if seed is not None else random.randint(0, 2**32 - 1)
    print(f"Prompt: {final_prompt}")
    print(f"Base Seed: {base_seed}")

    images: List[Image.Image] = []
    count = 4 if four_variations else 1
    for i in range(count):
        current_seed = base_seed + i
        generator = torch.Generator(device=device).manual_seed(current_seed)
        start = time.time()
        img = pipe(
            prompt=final_prompt,
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator,
        ).images[0]
        elapsed = time.time() - start
        variant_tag = f"v{i+1}" if four_variations else "single"
        print(f"Generated {variant_tag} (seed={current_seed}) in {elapsed:.2f}s")
        if save:
            fname = f"image_{int(time.time())}_{variant_tag}.png"
            out_path = out_dir / fname
            img.save(out_path)
            print(f"Saved: {out_path}")
            GenerationRecord.append((final_prompt, str(out_path), current_seed, elapsed))
        display(img)
        images.append(img)

    return images if four_variations else images[0]

def generate_and_display(
    prompts: List[str],
    width=1024,
    height=1024,
    steps=4,
    guidance=1.0,
    seed: Optional[int] = None,
    save: bool = True,
    variations_per_prompt: int = 1,
):
    """Generate images for a list of prompts.

    Args:
        prompts: List of fully prepared prompts (already prefixed if needed)
        width/height/steps/guidance: Generation parameters
        seed: Base seed for the first prompt; subsequent prompts and variations increment deterministically
        save: Save generated images to disk
        variations_per_prompt: Number of images to generate per prompt (e.g., 4 for four variations)
    """
    if not prompts:
        raise ValueError("No prompts provided.")
    pipe = load_pipeline(base_model, selected_lora)
    out_dir = cfg.output_dir / "generated_images"
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {out_dir}")

    gen_seed = seed if seed is not None else random.randint(0, 2**32 - 1)
    print(f"Initial base seed: {gen_seed}")

    for idx, prompt in enumerate(prompts, start=1):
        base_seed_for_prompt = gen_seed + (idx - 1)
        print(f"Prompt {idx}/{len(prompts)}: {prompt}")
        for v in range(max(1, int(variations_per_prompt))):
            current_seed = base_seed_for_prompt + v
            print(f"  Variation {v+1}/{variations_per_prompt} (seed={current_seed})")
            generator = torch.Generator(device=device).manual_seed(current_seed)
            start = time.time()
            image = pipe(
                prompt=prompt,
                width=width,
                height=height,
                num_inference_steps=steps,
                guidance_scale=guidance,
                generator=generator,
            ).images[0]
            elapsed = time.time() - start
            fname = f"image_{int(time.time())}_{idx:03d}_v{v+1}.png"
            out_path = out_dir / fname
            if save:
                image.save(out_path)
            display(image)
            GenerationRecord.append((prompt, str(out_path), current_seed, elapsed))
    print("Generation complete.")
    return GenerationRecord

In [ ]:
imgs = generate_single(
    "photo of alicegirl  child reading alphabet book",
    four_variations=True,
    seed=12345,
    skip_preprocess=True
)

In [ ]:
# 6) Batch Inference with Simple Loop (No DataLoader for Generation)

# Build final prompts prefixed with trigger word from config
final_prompts = prefix_prompts(user_prompts, cfg.trigger_word)
print("First 3 prompts:", final_prompts[:3])

# Run generation on all prompts (adjust steps/guidance as needed)
_ = generate_and_display(final_prompts, width=1024, height=1024, steps=4, guidance=1.0, seed=None)

In [ ]:
# 7) Postprocessing and Decoding (Minimal for Image Generation)

# For text-to-image we don't classify; we simply record metadata.
# Display the generation record in a small table-like print.
for row in GenerationRecord:
    prompt, path, seed_used, elapsed = row
    print(f"prompt='{prompt[:40]}...' seed={seed_used} time={elapsed:.2f}s file={path}")

print(f"Total images: {len(GenerationRecord)}")

In [ ]:
# 8) Performance Benchmarking (Latency and Throughput)

from statistics import median

def benchmark_pipeline(prompt: str, runs: int = 5, steps: int = 4):
    pipe = load_pipeline(base_model, selected_lora)
    times = []
    for i in range(runs):
        gen = torch.Generator(device=device).manual_seed(1234 + i)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        _ = pipe(prompt=prompt, width=1024, height=1024, num_inference_steps=steps, guidance_scale=1.0, generator=gen).images[0]
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.time() - t0)
    times_sorted = sorted(times)
    p50 = times_sorted[len(times_sorted)//2]
    p95 = times_sorted[max(0, int(len(times_sorted)*0.95)-1)]
    tput = 1.0 / (sum(times)/len(times)) if times else 0.0
    return {"runs": runs, "times": times, "p50": p50, "p95": p95, "throughput_img_per_s": tput}

if user_prompts:
    bench = benchmark_pipeline(prefix_prompts([user_prompts[0]], cfg.trigger_word)[0], runs=3, steps=4)
    print("Benchmark:", bench)
else:
    print("No prompts available for benchmarking.")

In [ ]:
# 9) Optional: Export to ONNX and Validate Parity (Placeholder)

# Diffusers pipelines for FLUX are complex; exporting to ONNX is non-trivial and
# typically not recommended for rapid iteration. This cell is a placeholder to show
# how you *might* attempt component export (UNet/text encoder) if desired.
# We skip actual export to keep notebook lightweight.

print("ONNX export skipped (placeholder). Components of FLUX pipeline not exported.")

In [ ]:
# 10) Save Predictions to Disk (CSV)

import csv

csv_path = cfg.output_dir / "generated_images" / "inference_log.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["prompt", "path", "seed", "elapsed_sec"])
    for row in GenerationRecord:
        writer.writerow(list(row))

print("Wrote:", csv_path)

In [ ]:
# 11) Basic Unit Tests for Inference Pipeline

def test_pipeline_loaded():
    pipe = load_pipeline(base_model, selected_lora)
    assert pipe is not None, "Pipeline failed to load"

def test_generation_record_shape():
    for entry in GenerationRecord:
        assert len(entry) == 4, "GenerationRecord entry should have 4 fields"

def test_images_exist():
    for _, path, _, _ in GenerationRecord:
        assert Path(path).exists(), f"Missing generated image: {path}"

def run_tests():
    failures = []
    for fn in [test_pipeline_loaded, test_generation_record_shape, test_images_exist]:
        try:
            fn()
            print(f"PASS: {fn.__name__}")
        except AssertionError as e:
            print(f"FAIL: {fn.__name__} -> {e}")
            failures.append((fn.__name__, str(e)))
    if failures:
        print("Tests completed with failures.")
    else:
        print("All tests passed.")

run_tests()